In [1]:
import numpy as np
import pandas as pd

# UF Dataset

In [3]:
import pandas as pd
import glob
import os

# Path to folder containing *_all_split.csv files
path = "/orange/ruogu.fang/tienyuchang/OCTRFF_Data/data/UF-cohort/IRB2024_v5_all/split/tune8-eval2/"  # change this path if your CSVs are elsewhere
#path = "/blue/ruogu.fang/tienyuchang/IRB2024v5_ADCON_DL_data/"

# Find all *_all_split.csv files
files = glob.glob(os.path.join(path, "*_all_split.csv"))
#files = glob.glob(os.path.join(path, "*_detect_data.csv"))

all_summaries = []

for file in files:
    #dataset_name = os.path.basename(file).replace("_all_split.csv", "")
    dataset_name = os.path.basename(file).replace("_detect_data.csv", "")
    print(f"Processing {dataset_name} ...")

    df = pd.read_csv(file)
    
    # Ensure required columns exist, person_id
    df['split'] = df['split'].map({'train':'Train','val':'Validation','test':'Test'})
    df['sub_category'] = df['label']
    df = df.rename(columns={"person_id":"patient_id"})

    # Count images and unique patients per sub_category & split
    summary = (
        df.groupby(["sub_category", "split"])
          .agg(images=("OCT", "count"),
               patients=("patient_id", pd.Series.nunique))
          .reset_index()
    )

    # Pivot by split for both images and patients
    pivot_images = summary.pivot(index="sub_category", columns="split", values="images").fillna(0).astype(int)
    pivot_patients = summary.pivot(index="sub_category", columns="split", values="patients").fillna(0).astype(int)

    # Add total counts
    pivot_images["Total_images"] = pivot_images.sum(axis=1)
    pivot_patients["Total"] = pivot_patients.sum(axis=1)

    # Calculate image percentage
    pivot_images["Percent"] = (pivot_images["Total_images"] / pivot_images["Total_images"].sum() * 100).round(2)

    # Merge both tables
    combined = pd.concat([pivot_images, pivot_patients.add_suffix("_patients")], axis=1)

    # Format total images with percentages
    combined["Total_images"] = combined.apply(lambda x: f"{x['Total_images'].astype(int)}({x['Percent']}%)", axis=1)
    combined = combined.drop(columns=["Percent"]).reset_index()

    # Add dataset name
    combined.insert(0, "Dataset name", dataset_name)

    # Add "in total" row
    total_row = {
        "Dataset name": dataset_name,
        "sub_category": "in total:",
        "Train": pivot_images["Train"].sum(),
        "Validation": pivot_images["Validation"].sum(),
        "Test": pivot_images["Test"].sum(),
        "Total_images": pivot_images["Total_images"].sum(),
        "Train_patients": pivot_patients["Train"].sum(),
        "Validation_patients": pivot_patients["Validation"].sum(),
        "Test_patients": pivot_patients["Test"].sum(),
        "Total_patients": pivot_patients["Total"].sum(),
    }
    combined = pd.concat([combined, pd.DataFrame([total_row])], ignore_index=True)

    all_summaries.append(combined)

# Combine all datasets into one CSV
final_df = pd.concat(all_summaries, ignore_index=True)
output_path = "summary_all_datasets.csv"
final_df.to_csv(output_path, index=False)

print(f"\n✅ Summary saved to: {output_path}")

Processing Glaucoma_all_split.csv ...
Processing CRVO_CRAO_all_split.csv ...
Processing DR_all_split.csv ...
Processing RNV_all_split.csv ...
Processing DME_all_split.csv ...
Processing MH_all_split.csv ...
Processing AMD_all_split.csv ...
Processing ERM_all_split.csv ...
Processing Cataract_all_split.csv ...
Processing Drusen_all_split.csv ...
Processing Glaucoma_binary_all_split.csv ...
Processing PVD_all_split.csv ...
Processing CSR_all_split.csv ...
Processing DR_binary_all_split.csv ...
Processing DME_binary_all_split.csv ...

✅ Summary saved to: summary_all_datasets.csv


## Demographic

In [19]:
tmp = '/orange/ruogu.fang/tienyuchang/OCTRFF_Data/data/UF-cohort/IRB2024_v5_all/split/tune8-eval2/DME_binary_all_split.csv'
df = pd.read_csv(tmp)
df.head() #no demographic

,Unnamed: 0,folder,imgname,eye,slice_indices,slice_num,fundus_imgname,patient_id,oct_imgname,OCT,depth,oct_valid,oct_img_size,fundus_valid,fundus_img_size,label,icd_eye,DME_icd,split
0,91101,1.2.840.114158.5158297108704163481144891534268...,1.2.840.114158.5158297108704163481144891534268...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.4846447641128135749678241434204...,711799721801,1.2.840.114158.5158297108704163481144891534268...,1.2.840.114158.5158297108704163481144891534268...,25,True,"(512, 496)",True,"(768, 768)",0,NaN,NaN,train
1,39422,1.2.840.114158.5564532448905775614105867282945...,1.2.840.114158.5564532448905775614105867282945...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5381874925697122863145507444135...,711799987396,1.2.840.114158.5564532448905775614105867282945...,1.2.840.114158.5564532448905775614105867282945...,25,True,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train
2,92119,1.2.840.114158.5548606368008319176105494651568...,1.2.840.114158.5548606368008319176105494651568...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5462627551523412824139273758867...,711802739250,1.2.840.114158.5548606368008319176105494651568...,1.2.840.114158.5548606368008319176105494651568...,25,True,"(512, 496)",True,"(768, 768)",0,NaN,NaN,train
3,23444,1.2.840.114158.4993568100886566904179743235777...,1.2.840.114158.4993568100886566904179743235777...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.4879302780628152829761513677999...,711795682330,1.2.840.114158.4993568100886566904179743235777...,1.2.840.114158.4993568100886566904179743235777...,25,True,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train
4,15251,1.2.840.114158.4749549049777858817504316833813...,1.2.840.114158.4749549049777858817504316833813...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5365510814731594793158825115291...,711795244287,1.2.840.114158.4749549049777858817504316833813...,1.2.840.114158.4749549049777858817504316833813...,25,True,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train


In [32]:
df['folder'].head()[0]

'1.2.840.114158.515829710870416348114489153426844694702'

In [34]:
pat_df = pd.read_csv('/blue/ruogu.fang/tienyuchang/OCT_EDA/OCT_pat_demographic.csv')
print(pat_df.shape)

(203208, 9)


In [35]:
pat_df["folder"] = pat_df["Path"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
pat_df.head()

,person_id,birth_datetime,gender_source_value,race_ethnicity,oct_date,DeidStudyInstanceUID,Path,age,age_group,folder
0,711800484204,1963-12-04 00:00:00,FEMALE,NHB,2019-12-17,1.2.840.114158.5295183688886106011151740589069...,/orange/ruogu.fang/tienyuchang/IRB202400720_Da...,56.0,45-64,1.2.840.114158.5014346414258794812480121134468...
1,711800484204,1963-12-04 00:00:00,FEMALE,NHB,2019-12-17,1.2.840.114158.5295183688886106011151740589069...,/orange/ruogu.fang/tienyuchang/IRB202400720_Da...,56.0,45-64,1.2.840.114158.4651951072651510402117886196374...
2,711801142647,1945-05-19 00:00:00,MALE,NHB,2023-09-13,1.2.840.114158.5247150866857322362147182743912...,/orange/ruogu.fang/tienyuchang/IRB202400720_Da...,78.0,>=75,1.2.840.114158.5401087116455391149162375612181...
3,711801142647,1945-05-19 00:00:00,MALE,NHB,2023-09-13,1.2.840.114158.5247150866857322362147182743912...,/orange/ruogu.fang/tienyuchang/IRB202400720_Da...,78.0,>=75,1.2.840.114158.5490959419731094659177497866886...
4,711801142647,1945-05-19 00:00:00,MALE,NHB,2023-04-09,1.2.840.114158.4716105751388694190161709055419...,/orange/ruogu.fang/tienyuchang/IRB202400720_Da...,77.0,>=75,1.2.840.114158.5441599179359305018894945332734...


In [36]:
pat_df['Path'].head()[0]

'/orange/ruogu.fang/tienyuchang/IRB202400720_Data/Images/1.2.840.114158.529518368888610601115174058906982813602/1.2.840.114158.5666974157110624854661169970293646001/1.2.840.114158.50143464142587948124801211344680148609.dcm'

In [37]:
df_demo = df.merge(pat_df[['gender_source_value','race_ethnicity','age','folder']],on='folder')
df_demo.head()

,Unnamed: 0,folder,imgname,eye,slice_indices,slice_num,fundus_imgname,patient_id,oct_imgname,OCT,...,oct_img_size,fundus_valid,fundus_img_size,label,icd_eye,DME_icd,split,gender_source_value,race_ethnicity,age
0,91101,1.2.840.114158.5158297108704163481144891534268...,1.2.840.114158.5158297108704163481144891534268...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.4846447641128135749678241434204...,711799721801,1.2.840.114158.5158297108704163481144891534268...,1.2.840.114158.5158297108704163481144891534268...,...,"(512, 496)",True,"(768, 768)",0,NaN,NaN,train,FEMALE,NHW,67.0
1,39422,1.2.840.114158.5564532448905775614105867282945...,1.2.840.114158.5564532448905775614105867282945...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5381874925697122863145507444135...,711799987396,1.2.840.114158.5564532448905775614105867282945...,1.2.840.114158.5564532448905775614105867282945...,...,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train,MALE,NHW,68.0
2,92119,1.2.840.114158.5548606368008319176105494651568...,1.2.840.114158.5548606368008319176105494651568...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5462627551523412824139273758867...,711802739250,1.2.840.114158.5548606368008319176105494651568...,1.2.840.114158.5548606368008319176105494651568...,...,"(512, 496)",True,"(768, 768)",0,NaN,NaN,train,FEMALE,NHW,60.0
3,23444,1.2.840.114158.4993568100886566904179743235777...,1.2.840.114158.4993568100886566904179743235777...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.4879302780628152829761513677999...,711795682330,1.2.840.114158.4993568100886566904179743235777...,1.2.840.114158.4993568100886566904179743235777...,...,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train,MALE,HISPANIC,68.0
4,15251,1.2.840.114158.4749549049777858817504316833813...,1.2.840.114158.4749549049777858817504316833813...,latR,0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-18...,25,1.2.840.114158.5365510814731594793158825115291...,711795244287,1.2.840.114158.4749549049777858817504316833813...,1.2.840.114158.4749549049777858817504316833813...,...,"(512, 496)",True,"(768, 768)",1,right,E11.3511,train,MALE,NHB,56.0


In [41]:
import pandas as pd
import glob
import os

# Path to folder containing *_all_split.csv files
path = "/orange/ruogu.fang/tienyuchang/OCTRFF_Data/data/UF-cohort/IRB2024_v5_all/split/tune8-eval2/"

# Find all *_all_split.csv files
files = glob.glob(os.path.join(path, "*_all_split.csv"))
# files = glob.glob(os.path.join(path, "*_detect_data.csv"))

all_dataset_summaries = []
all_cohort_rows = []   # store all rows from all files (later dedup by folder)

for file in files:
    dataset_name = os.path.basename(file).replace("_all_split.csv", "").replace(".csv", "")
    print(f"Processing {dataset_name} ...")

    df = pd.read_csv(file)
    df = df.merge(pat_df[['gender_source_value','race_ethnicity','age','folder']],on='folder')

    # Drop useless column if exists
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    # Normalize split names
    df["split"] = df["split"].map({"train": "Train", "val": "Validation", "test": "Test"}).fillna(df["split"])

    # Standardize patient_id column if needed
    if "person_id" in df.columns and "patient_id" not in df.columns:
        df = df.rename(columns={"person_id": "patient_id"})

    # Label -> sub_category
    df["sub_category"] = df["label"]

    # Fill missing demographic values
    if "gender_source_value" not in df.columns:
        df["gender_source_value"] = "Unknown"
    df["gender_source_value"] = df["gender_source_value"].fillna("Unknown")

    if "race_ethnicity" not in df.columns:
        df["race_ethnicity"] = "Unknown"
    df["race_ethnicity"] = df["race_ethnicity"].fillna("Unknown")

    # --------------------------------------------
    # Save for full cohort pooling later
    # --------------------------------------------
    all_cohort_rows.append(df)

    # --------------------------------------------
    # Dataset-level DEMOGRAPHICS (by sub_category + split)
    # --------------------------------------------

    # ---- Gender summary (unique patients) ----
    gender_summary = (
        df.groupby(["sub_category", "split", "gender_source_value"])
          .agg(patients=("patient_id", pd.Series.nunique))
          .reset_index()
    )

    gender_pivot = (
        gender_summary.pivot(index=["sub_category", "gender_source_value"],
                             columns="split", values="patients")
        .fillna(0)
        .astype(int)
    )

    for col in ["Train", "Validation", "Test"]:
        if col not in gender_pivot.columns:
            gender_pivot[col] = 0
    gender_pivot = gender_pivot[["Train", "Validation", "Test"]]

    gender_pivot["Total_patients"] = gender_pivot.sum(axis=1)

    gender_table = gender_pivot.reset_index()
    gender_table.insert(0, "Dataset name", dataset_name)
    gender_table.insert(2, "Demographic", "Gender")
    gender_table = gender_table.rename(columns={"gender_source_value": "Group"})

    # add in-total row per label
    gender_total_rows = []
    for lab in gender_table["sub_category"].unique():
        sub = gender_table[gender_table["sub_category"] == lab]
        gender_total_rows.append({
            "Dataset name": dataset_name,
            "sub_category": lab,
            "Demographic": "Gender",
            "Group": "in total:",
            "Train": int(sub["Train"].sum()),
            "Validation": int(sub["Validation"].sum()),
            "Test": int(sub["Test"].sum()),
            "Total_patients": int(sub["Total_patients"].sum())
        })
    gender_table = pd.concat([gender_table, pd.DataFrame(gender_total_rows)], ignore_index=True)

    # ---- Race summary (unique patients) ----
    race_summary = (
        df.groupby(["sub_category", "split", "race_ethnicity"])
          .agg(patients=("patient_id", pd.Series.nunique))
          .reset_index()
    )

    race_pivot = (
        race_summary.pivot(index=["sub_category", "race_ethnicity"],
                           columns="split", values="patients")
        .fillna(0)
        .astype(int)
    )

    for col in ["Train", "Validation", "Test"]:
        if col not in race_pivot.columns:
            race_pivot[col] = 0
    race_pivot = race_pivot[["Train", "Validation", "Test"]]

    race_pivot["Total_patients"] = race_pivot.sum(axis=1)

    race_table = race_pivot.reset_index()
    race_table.insert(0, "Dataset name", dataset_name)
    race_table.insert(2, "Demographic", "Race/Ethnicity")
    race_table = race_table.rename(columns={"race_ethnicity": "Group"})

    # add in-total row per label
    race_total_rows = []
    for lab in race_table["sub_category"].unique():
        sub = race_table[race_table["sub_category"] == lab]
        race_total_rows.append({
            "Dataset name": dataset_name,
            "sub_category": lab,
            "Demographic": "Race/Ethnicity",
            "Group": "in total:",
            "Train": int(sub["Train"].sum()),
            "Validation": int(sub["Validation"].sum()),
            "Test": int(sub["Test"].sum()),
            "Total_patients": int(sub["Total_patients"].sum())
        })
    race_table = pd.concat([race_table, pd.DataFrame(race_total_rows)], ignore_index=True)

    # ---- Age summary (per label + split) ----
    age_df = df[["sub_category", "split", "patient_id", "age"]].copy()
    age_df = age_df.dropna(subset=["age"])

    age_summary = (
        age_df.groupby(["sub_category", "split"])
              .agg(
                  N_patients=("patient_id", pd.Series.nunique),
                  age_mean=("age", "mean"),
                  age_std=("age", "std"),
                  age_median=("age", "median"),
                  age_min=("age", "min"),
                  age_max=("age", "max")
              )
              .reset_index()
    )

    for c in ["age_mean", "age_std", "age_median", "age_min", "age_max"]:
        age_summary[c] = age_summary[c].round(2)

    age_summary.insert(0, "Dataset name", dataset_name)
    age_summary.insert(2, "Demographic", "Age")

    # Combine all dataset demographic tables
    dataset_final = pd.concat([gender_table, race_table, age_summary], ignore_index=True)
    all_dataset_summaries.append(dataset_final)


# ======================================================
# ✅ FULL COHORT DEMOGRAPHIC (drop duplicate folder)
# ======================================================
cohort_df = pd.concat(all_cohort_rows, ignore_index=True)

if "folder" not in cohort_df.columns:
    raise ValueError("Column 'folder' not found, cannot drop duplicate folder for cohort summary.")

# Drop duplicate folder (keep first)
cohort_df = cohort_df.drop_duplicates(subset=["folder"]).copy()

# Ensure patient_id exists
if "person_id" in cohort_df.columns and "patient_id" not in cohort_df.columns:
    cohort_df = cohort_df.rename(columns={"person_id": "patient_id"})

# Normalize split names again
cohort_df["split"] = cohort_df["split"].map({"train": "Train", "val": "Validation", "test": "Test"}).fillna(cohort_df["split"])

# Fill missing demographics
cohort_df["gender_source_value"] = cohort_df.get("gender_source_value", "Unknown").fillna("Unknown")
cohort_df["race_ethnicity"] = cohort_df.get("race_ethnicity", "Unknown").fillna("Unknown")

# Cohort overview counts
total_folders = cohort_df["folder"].nunique()
total_patients = cohort_df["patient_id"].nunique()

cohort_overview = pd.DataFrame([{
    "Dataset name": "FULL_COHORT",
    "Demographic": "Overview",
    "Metric": "Total folders (dedup)",
    "Value": total_folders
}, {
    "Dataset name": "FULL_COHORT",
    "Demographic": "Overview",
    "Metric": "Total patients",
    "Value": total_patients
}])

# ---- Cohort gender distribution (unique patients) ----
gender_cohort = (
    cohort_df.groupby(["gender_source_value"])
             .agg(patients=("patient_id", pd.Series.nunique))
             .reset_index()
             .rename(columns={"gender_source_value": "Group"})
)
gender_cohort["Percent"] = (gender_cohort["patients"] / gender_cohort["patients"].sum() * 100).round(2)

gender_cohort.insert(0, "Dataset name", "FULL_COHORT")
gender_cohort.insert(1, "Demographic", "Gender")

# ---- Cohort race distribution (unique patients) ----
race_cohort = (
    cohort_df.groupby(["race_ethnicity"])
             .agg(patients=("patient_id", pd.Series.nunique))
             .reset_index()
             .rename(columns={"race_ethnicity": "Group"})
)
race_cohort["Percent"] = (race_cohort["patients"] / race_cohort["patients"].sum() * 100).round(2)

race_cohort.insert(0, "Dataset name", "FULL_COHORT")
race_cohort.insert(1, "Demographic", "Race/Ethnicity")

# ---- Cohort age stats (unique patients) ----
age_cohort = cohort_df[["patient_id", "age"]].dropna(subset=["age"]).drop_duplicates(subset=["patient_id"])

age_stats = pd.DataFrame([{
    "Dataset name": "FULL_COHORT",
    "Demographic": "Age",
    "N_patients": int(age_cohort["patient_id"].nunique()),
    "age_mean": round(age_cohort["age"].mean(), 2),
    "age_std": round(age_cohort["age"].std(), 2),
    "age_median": round(age_cohort["age"].median(), 2),
    "age_min": round(age_cohort["age"].min(), 2),
    "age_max": round(age_cohort["age"].max(), 2),
}])


# ======================================================
# ✅ Save outputs
# ======================================================
dataset_out = "demographic_summary_all_datasets.csv"
final_dataset_df = pd.concat(all_dataset_summaries, ignore_index=True)
final_dataset_df.to_csv(dataset_out, index=False)
print(f"\n✅ Dataset demographic summary saved to: {dataset_out}")

cohort_out = "demographic_summary_full_cohort.csv"
full_cohort_df = pd.concat([cohort_overview, gender_cohort, race_cohort, age_stats], ignore_index=True)
full_cohort_df.to_csv(cohort_out, index=False)
print(f"✅ Full cohort demographic summary saved to: {cohort_out}")


Processing Glaucoma ...
Processing CRVO_CRAO ...
Processing DR ...
Processing RNV ...
Processing DME ...
Processing MH ...
Processing AMD ...
Processing ERM ...
Processing Cataract ...
Processing Drusen ...
Processing Glaucoma_binary ...
Processing PVD ...
Processing CSR ...
Processing DR_binary ...
Processing DME_binary ...

✅ Dataset demographic summary saved to: demographic_summary_all_datasets.csv
✅ Full cohort demographic summary saved to: demographic_summary_full_cohort.csv


# Public Dataset

In [15]:

def summary_dataset(name, path, files):
    all_summaries = []

    for file in files:
        dataset_name = os.path.basename(file)
        dataset_name = dataset_name.replace("_all_split.csv", "").replace(".csv", "")

        print(f"Processing {dataset_name} ...")

        df = pd.read_csv(file)

        # Normalize split names (keeps unknown splits unchanged)
        df["split"] = df["split"].map({
            "train": "Train",
            "val": "Validation",
            "test": "Test"
        }).fillna(df["split"])

        df["sub_category"] = df["label"]

        # Count images per sub_category & split
        summary = (
            df.groupby(["sub_category", "split"])
              .agg(images=("image", "count"))
              .reset_index()
        )

        # Pivot
        pivot_images = (
            summary.pivot(index="sub_category", columns="split", values="images")
                   .fillna(0)
                   .astype(int)
        )

        # Ensure missing columns exist
        for col in ["Train", "Validation", "Test"]:
            if col not in pivot_images.columns:
                pivot_images[col] = 0

        pivot_images = pivot_images[["Train", "Validation", "Test"]]

        # Numeric totals
        pivot_images["Total_images_num"] = pivot_images.sum(axis=1)

        # Percentage
        total_all = pivot_images["Total_images_num"].sum()
        pivot_images["Percent"] = (pivot_images["Total_images_num"] / total_all * 100).round(2)

        # Final formatted total column (string)
        pivot_images["Total_images"] = pivot_images.apply(
            lambda r: f"{int(r['Total_images_num'])}({r['Percent']}%)",
            axis=1
        )

        # Build final table (drop helper numeric columns)
        combined = pivot_images.drop(columns=["Total_images_num", "Percent"]).reset_index()

        # Add dataset info
        combined.insert(0, "Dataset name", name)
        combined.insert(1, "Task", dataset_name)

        # Add total row (correct numeric sums)
        total_row = {
            "Dataset name": name,
            "Task": dataset_name,
            "sub_category": "in total:",
            "Train": int(pivot_images["Train"].sum()),
            "Validation": int(pivot_images["Validation"].sum()),
            "Test": int(pivot_images["Test"].sum()),
            "Total_images": int(pivot_images["Total_images"].str.extract(r"(\d+)")[0].astype(int).sum())
        }

        combined = pd.concat([combined, pd.DataFrame([total_row])], ignore_index=True)

        all_summaries.append(combined)

    final_df = pd.concat(all_summaries, ignore_index=True)
    output_path = os.path.join(f"{name}_summary_all_datasets.csv")
    final_df.to_csv(output_path, index=False)

    print(f"\n✅ Summary saved to: {output_path}")

In [16]:
name = "CellData"
path = "/orange/ruogu.fang/tienyuchang/CellData/OCT"
files = glob.glob(os.path.join(path, "*_all.csv"))
#files = glob.glob(os.path.join(path, "*_detect_data.csv"))
summary_dataset(name,path,files)

Processing DME_all ...
Processing CNV_all ...
Processing DRUSEN_all ...

✅ Summary saved to: CellData_summary_all_datasets.csv


In [17]:
name = "OCTDL"
path = "/orange/ruogu.fang/tienyuchang/OCTDL"
files = glob.glob(os.path.join(path, "*_all.csv"))
#files = glob.glob(os.path.join(path, "*_detect_data.csv"))
summary_dataset(name,path,files)

Processing AMD_all ...
Processing DME_all ...

✅ Summary saved to: OCTDL_summary_all_datasets.csv


In [18]:
name = "OCTID"
path = "/orange/ruogu.fang/tienyuchang/OCTID"
files = glob.glob(os.path.join(path, "*_all.csv"))
#files = glob.glob(os.path.join(path, "*_detect_data.csv"))
summary_dataset(name,path,files)

Processing DR_all ...
Processing CSR_all ...
Processing MH_all ...

✅ Summary saved to: OCTID_summary_all_datasets.csv
